# 第 07 章：StateGraph 基础——State、Reducer、Node、Edge 与显式 ReAct（离线工程实验）

**目标**：执行本章稳定公共契约并观察失败护栏。  
**环境与预计用时**：Python 3.12、offline profile，约 15–25 分钟。  
本 Notebook 由同名 Markdown 中带 `sync` 标识的实验代码生成；可复用业务逻辑始终从 `mini_deerflow` package 导入。

## 1. offline profile 初始化

显式选择离线模型档位，基础实验不得读取供应商 Key。

In [1]:
from mini_deerflow.config import ModelProfile, ModelSettings

lesson_settings = ModelSettings(profile=ModelProfile.OFFLINE)
assert lesson_settings.profile is ModelProfile.OFFLINE


## 2. 前置能力探针

验证当前 kernel 使用课程锁定的主版本，并能导入 Mini DeerFlow。

In [2]:
from importlib.metadata import version
import mini_deerflow

assert version('langchain').startswith('1.3.')
assert version('langgraph').startswith('1.2.')
assert mini_deerflow.__file__


## 3. 最小成功实验

以下单元来自 Markdown 的稳定 sync marker。

### 实验 `ch07-explicit-react`

In [3]:
from langchain_core.messages import AIMessage, ToolMessage
from mini_deerflow.graph import create_explicit_react_graph
from mini_deerflow.models import create_offline_model
from mini_deerflow.tools import calculator

react_model = create_offline_model(
    [
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "calculator",
                    "args": {"operation": "multiply", "left": 6, "right": 7},
                    "id": "calc-42",
                    "type": "tool_call",
                }
            ],
        ),
        AIMessage(content="结果是 42。"),
    ]
)
react_graph = create_explicit_react_graph(model=react_model, tools=[calculator])
react_result = react_graph.invoke({"messages": [("user", "计算 6 × 7")]})

assert [event.as_text() for event in react_result["node_trace"]] == [
    "model",
    "tools",
    "model",
]
assert react_result["messages"][-1].content == "结果是 42。"
assert next(
    message.content
    for message in react_result["messages"]
    if isinstance(message, ToolMessage)
) == "42.0"


## 4. 状态/事件观察

观察消息、结构化对象、检索命中或 v2 event；不要只看最终自然语言。

### 实验 `ch07-stream-updates`

In [4]:
stream_model = create_offline_model(
    [
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "calculator",
                    "args": {"operation": "add", "left": 1, "right": 2},
                    "id": "calc-3",
                    "type": "tool_call",
                }
            ],
        ),
        AIMessage(content="3"),
    ]
)
stream_graph = create_explicit_react_graph(model=stream_model, tools=[calculator])
react_updates = list(
    stream_graph.stream(
        {"messages": [("user", "1 + 2")]},
        stream_mode="updates",
    )
)

assert [next(iter(update)) for update in react_updates] == ["model", "tools", "model"]
assert [
    event.as_text() for event in react_updates[1]["tools"]["node_trace"]
] == ["tools"]


## 5. 失败实验

失败必须被捕获并断言，证明护栏真的阻止了错误路径。

### 实验 `ch07-loop-limit-failure`

In [5]:
from langgraph.errors import GraphRecursionError

loop_messages = [
    AIMessage(
        content="",
        tool_calls=[
            {
                "name": "calculator",
                "args": {"operation": "add", "left": 1, "right": 1},
                "id": f"loop-{index}",
                "type": "tool_call",
            }
        ],
    )
    for index in range(10)
]
loop_graph = create_explicit_react_graph(
    model=create_offline_model(loop_messages),
    tools=[calculator],
)
try:
    loop_graph.invoke(
        {"messages": [("user", "持续调用工具")]},
        config={"recursion_limit": 3},
    )
except GraphRecursionError as error:
    loop_error = error
else:
    raise AssertionError("无界工具循环必须被 recursion_limit 终止")

assert "Recursion limit" in str(loop_error)


## 6. Mini DeerFlow 工程调用

以上实验只从 `mini_deerflow` 导入公共接口；Notebook 不复制 Agent、Tool 或 Schema 实现。

## 7. 分层练习

完成同名 Markdown 的练习 A（单点修改）、B（边界判断）、C（项目扩展）和延迟回忆题。先自行作答，再运行对应 pytest 获取即时反馈。

## 8. 自动验收摘要

在项目根目录运行 `make test`。本 Notebook 的所有代码单元必须有执行计数、不得保存 error output，教程验证结果不得出现本章 drift。

## 9. 清理临时资源

当前实验使用内存对象与 `TemporaryDirectory`，退出上下文后自动清理；不要把 API Key、向量库或临时产物写回仓库。